In [ ]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DO AGENTE PLANAPP
# ============================================================
#%pip install "mcp"
import importlib
import agent_jupyter

agent_jupyter = importlib.reload(agent_jupyter)

PlanAppAgent = agent_jupyter.PlanAppAgent

In [ ]:
import asyncio
import html
import ipywidgets as widgets

from IPython.display import display

from agent_jupyter import PlanAppAgent


# ============================================================
# INTERFACE
# ============================================================

titulo = widgets.HTML(
    value="""
    <h2 style="margin-bottom:5px;">
        📡 PlanApp AI — Análise de Enlace
    </h2>
    """
)

entrada = widgets.Textarea(
    placeholder=(
        "Exemplo:\n"
        "Analise um enlace entre a Praça da Sé e o Largo Treze em São Paulo."
    ),
    layout=widgets.Layout(
        width="100%",
        height="100px"
    )
)

botao_analisar = widgets.Button(
    description="🚀 Analisar enlace",
    button_style="primary",
    icon="play"
)

botao_nova = widgets.Button(
    description="🔄 Nova análise",
    button_style="",
    icon="refresh"
)

status = widgets.HTML(
    value="⚪ Aguardando análise..."
)

historico_status = widgets.HTML(
    value=""
)

log_execucao = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        max_height="350px",
        overflow="auto",
        border="1px solid #ddd",
        padding="8px",
        margin="10px 0 0 0"
    )
)


# ============================================================
# MAPA
#
# IMPORTANTE:
# Não usamos widgets.Output para o mapa.
# O próprio ipyleaflet.Map será colocado como filho
# de um VBox.
# ============================================================

mapa_output = widgets.VBox(
    layout=widgets.Layout(
        width="100%",
        min_height="600px",
        margin="15px 0 0 0"
    )
)


resposta = widgets.HTML(
    value=""
)


# ============================================================
# ESTADO
# ============================================================

executando = False
task_atual = None


# ============================================================
# CALLBACK DE STATUS
# ============================================================

def atualizar_status(mensagem):

    global historico_status

    status.value = (
        f"""
        <div style="
            padding:10px;
            margin-top:10px;
            border-radius:6px;
            background:#f5f5f5;
        ">
            {html.escape(str(mensagem))}
        </div>
        """
    )

    historico_status.value = (
        historico_status.value
        + f"<div>{html.escape(str(mensagem))}</div>"
    )


# ============================================================
# AGENTE
# ============================================================

agent = PlanAppAgent(
    progress_callback=atualizar_status
)


# ============================================================
# EXECUÇÃO ASSÍNCRONA
# ============================================================

async def executar_analise_async():

    global executando

    try:

        texto = entrada.value.strip()

        if not texto:

            status.value = (
                """
                <div style="
                    padding:10px;
                    color:#b00020;
                    background:#ffebee;
                    border-radius:6px;
                ">
                    ⚠️ Digite uma solicitação para iniciar a análise.
                </div>
                """
            )

            return


        # ----------------------------------------------------
        # LIMPA RESULTADOS ANTERIORES
        # ----------------------------------------------------

        resposta.value = ""

        mapa_output.children = []

        historico_status.value = ""

        with log_execucao:

            print("")
            print("=" * 70)
            print("🔄 Execução do agente")
            print("=" * 70)


        # ----------------------------------------------------
        # EXECUTA O AGENTE
        # ----------------------------------------------------

        resultado = await agent.ask(texto)


        # ----------------------------------------------------
        # DEBUG
        # ----------------------------------------------------

        print(
            "DEBUG MAPA:",
            type(agent.map),
            agent.map is not None
        )


        # ----------------------------------------------------
        # RENDERIZA O MAPA
        #
        # O Map é inserido diretamente no VBox.
        # Não usamos display() dentro de Output.
        # ----------------------------------------------------

        if agent.map is not None:

            mapa_output.children = [
                agent.map
            ]

        else:

            mapa_output.children = []


        # ----------------------------------------------------
        # RESPOSTA DO AGENTE
        # ----------------------------------------------------

        if resultado is None:

            resposta.value = ""

        else:

            texto_resultado = str(resultado)

            resposta.value = (
                """
                <div style="
                    margin-top:15px;
                    padding:15px;
                    border:1px solid #ddd;
                    border-radius:8px;
                    background:#fafafa;
                ">
                    <h3>📝 Resultado da análise</h3>
                    <div style="
                        white-space:pre-wrap;
                        line-height:1.5;
                    ">
                """
                + html.escape(texto_resultado)
                + """
                    </div>
                </div>
                """
            )


        # ----------------------------------------------------
        # STATUS FINAL
        # ----------------------------------------------------

        status.value = (
            """
            <div style="
                padding:10px;
                margin-top:10px;
                border-radius:6px;
                background:#e8f5e9;
                color:#1b5e20;
            ">
                🟢 Análise concluída
            </div>
            """
        )


    except Exception as e:

        erro = f"{type(e).__name__}: {e}"

        status.value = (
            f"""
            <div style="
                padding:10px;
                margin-top:10px;
                border-radius:6px;
                background:#ffebee;
                color:#b00020;
            ">
                ❌ Erro durante execução do agente:
                <b>{html.escape(erro)}</b>
            </div>
            """
        )

        with log_execucao:

            print("")
            print(
                f"❌ Erro durante execução do agente: {erro}"
            )


    finally:

        executando = False

        botao_analisar.disabled = False
        botao_nova.disabled = False


# ============================================================
# BOTÃO ANALISAR
# ============================================================

def executar_analise(_):

    global executando
    global task_atual

    if executando:

        return

    executando = True

    botao_analisar.disabled = True
    botao_nova.disabled = True

    status.value = (
        """
        <div style="
            padding:10px;
            margin-top:10px;
            border-radius:6px;
            background:#fff8e1;
            color:#795548;
        ">
            🔵 Iniciando análise...
        </div>
        """
    )

    task_atual = asyncio.create_task(
        executar_analise_async()
    )


# ============================================================
# NOVA ANÁLISE
# ============================================================

def nova_analise(_):

    global executando

    if executando:

        return

    entrada.value = ""

    resposta.value = ""

    mapa_output.children = []

    historico_status.value = ""

    with log_execucao:

        print("")
        print("=" * 70)
        print("🔄 Nova análise")
        print("=" * 70)

    status.value = (
        """
        <div style="
            padding:10px;
            margin-top:10px;
            border-radius:6px;
            background:#f5f5f5;
        ">
            ⚪ Aguardando análise...
        </div>
        """
    )


# ============================================================
# EVENTOS
# ============================================================

botao_analisar.on_click(
    executar_analise
)

botao_nova.on_click(
    nova_analise
)


# ============================================================
# EXEMPLOS
# ============================================================

exemplos = widgets.HTML(
    value="""
    <div style="
        margin-top:15px;
        padding:12px;
        border-radius:8px;
        background:#f5f5f5;
    ">

        <b>Exemplos de solicitações:</b>

        <ul>
            <li>
                Analise um enlace entre a Praça da Sé
                e o Largo Treze em São Paulo.
            </li>

            <li>
                Analise um enlace entre a Vila Germânica
                e o Terminal Fonte em Blumenau.
            </li>

            <li>
                Analise um enlace entre Balneário Camboriú
                e Florianópolis.
            </li>
        </ul>

    </div>
    """
)


# ============================================================
# LAYOUT FINAL
# ============================================================

interface = widgets.VBox(
    [
        titulo,
        entrada,
        widgets.HBox(
            [
                botao_analisar,
                botao_nova
            ]
        ),
        status,
        historico_status,
        log_execucao,
        mapa_output,
        resposta,
        exemplos
    ],
    layout=widgets.Layout(
        width="100%"
    )
)


display(interface)

In [ ]:
# ============================================================
# CÉLULA 3 — MAPA DO ÚLTIMO ENLACE ANALISADO
# ============================================================

from map_utils import mostrar_mapa_enlace


try:

    mapa_enlace = mostrar_mapa_enlace(agent)

except Exception as e:

    print("❌ Não foi possível mostrar o mapa.")
    print()
    print(f"Erro: {e}")
    print()
    print(
        "Verifique se o agente já executou uma análise "
        "com dois pontos geocodificados."
    )

In [ ]:
import httpx

async def testar_mcp():
    url = "http://172.17.0.1:8010/mcp"

    print("🔎 Testando conexão com MCP...")
    print(url)

    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(url)

        print("🟢 HTTP respondeu")
        print("Status:", response.status_code)
        print("Headers:", dict(response.headers))
        print("Resposta:", response.text[:500])

    except Exception as e:
        print("❌ Erro na conexão MCP:")
        print(type(e).__name__)
        print(str(e))

await testar_mcp()

In [ ]:
import inspect
import agent_jupyter

print(inspect.getsource(agent_jupyter.PlanAppAgent.build_ollama_tools))